# 🔬 Chaos 2 Clarity — Notebook 00: Data Generation

**Paper:** *Chaos 2 Clarity: A Self-Improving Semantic Orchestration Framework for LLM-Driven BI*

This notebook generates the complete 3-source retail enterprise environment:
1. **PostgreSQL** (via DuckDB) — 14 tables, sales/orders
2. **Salesforce CRM export** — accounts, opportunities (CSV)
3. **Logistics CSV** — delivery events, carriers

**Properties (from paper Section 6):**
- 47 columns total
- Inconsistent naming: `line_value` for revenue (not `order_total`)
- No shared primary keys across sources
- Zero pre-existing documentation

## Setup

In [ ]:
# Install dependencies
!pip install -q duckdb pandas numpy faker chromadb google-generativeai openai scipy statsmodels scikit-learn matplotlib seaborn tqdm sentence-transformers networkx

In [ ]:
# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

# Set up project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/chaos2clarity'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/eval/questions', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/eval/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)
print(f'✅ Project directory: {PROJECT_DIR}')

In [ ]:
# Clone the repo (or upload src/ manually)
# !git clone https://github.com/bankupalliravi/chaos2clarity.git /content/chaos2clarity

# For now, add src to path
import sys
sys.path.insert(0, '/content/chaos2clarity')
# If running from the repo directly:
# sys.path.insert(0, '.')

## 1. Generate Synthetic Data

In [ ]:
from src.data_generator import setup_full_environment, get_schema_ddl, get_column_descriptions_str

# Generate all 3 sources
conn, sf_accounts, sf_opportunities, logistics, emails_df, tickets_df, status_codes = \
    setup_full_environment(data_dir=f'{PROJECT_DIR}/data', db_path=f'{PROJECT_DIR}/data/retail.duckdb')

print('\n' + '='*60)
print('DATA ENVIRONMENT READY')
print('='*60)

In [ ]:
# Verify: show schema
schema_ddl = get_schema_ddl(conn)
print('=== Schema DDL ===')
print(schema_ddl)
print()

# Save schema snapshot for reproducibility
with open(f'{PROJECT_DIR}/eval/schema_snapshot.sql', 'w') as f:
    f.write(schema_ddl)
print('✅ Schema snapshot saved')

In [ ]:
# Verify: sample data from each source
print('=== PostgreSQL: orders (note: revenue column is line_value) ===')
display(conn.execute('SELECT * FROM orders LIMIT 5').fetchdf())

print('\n=== Salesforce: accounts (note: email_address not email) ===')
display(conn.execute('SELECT * FROM sf_accounts LIMIT 5').fetchdf())

print('\n=== Logistics: deliveries (note: shipping_ref links to orders.shipping_id) ===')
display(conn.execute('SELECT * FROM logistics_deliveries LIMIT 5').fetchdf())

In [ ]:
# Verify: test a cross-source join (the implicit email link)
cross_source_test = conn.execute("""
    SELECT c.name, c.email, sa.account_name, sa.email_address, sa.sf_segment
    FROM customers c
    JOIN sf_accounts sa ON c.email = sa.email_address
    LIMIT 5
""").fetchdf()
print('=== Cross-source test: PG customers ↔ SF accounts (via email) ===')
display(cross_source_test)
print(f'\n✅ {len(cross_source_test)} cross-source matches found (sample)')

## 2. Build Gold Semantic Model

In [ ]:
from src.semantic_layer import build_gold_semantic_model
import json

gold_model = build_gold_semantic_model()

print(f'Gold Entities: {len(gold_model.entities)}')
print(f'Gold Metrics: {len(gold_model.metrics)}')
print(f'Gold Relationships: {len(gold_model.relationships)}')
print(f'Gold Policies: {len(gold_model.policies)}')

# Save
with open(f'{PROJECT_DIR}/eval/gold_semantic_model.json', 'w') as f:
    f.write(gold_model.to_json())
print('\n✅ Gold semantic model saved')

In [ ]:
# Display the gold model summary
print(gold_model.to_summary())

## 3. Create 50-Question BI Suite

Each question has:
- Natural language prompt
- Gold SQL
- Gold result set (from running gold SQL)
- Primary error class annotation

In [ ]:
import json

def create_question(qid, tier, nl_prompt, gold_sql, error_class, error_note=''):
    """Create a question dict, run gold SQL, capture result."""
    try:
        result = conn.execute(gold_sql).fetchall()
        columns = [desc[0] for desc in conn.description]
        rows = [dict(zip(columns, row)) for row in result]
        gold_result = {'rows': rows, 'columns': columns}
    except Exception as e:
        print(f'⚠️ Gold SQL failed for {qid}: {e}')
        gold_result = None
    
    return {
        'id': qid,
        'tier': tier,
        'nl_prompt': nl_prompt,
        'gold_sql': gold_sql,
        'gold_result': gold_result,
        'primary_error_class': error_class,
        'error_note': error_note,
    }

questions = []

In [ ]:
# ═══════════════════════════════════════════════
# L1 — Single-Source Metric (15 questions)
# ═══════════════════════════════════════════════

l1_questions = [
    ('L1-01', 'What was total gross revenue last quarter?',
     "SELECT SUM(line_value) as total_revenue FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL 90 DAY",
     'E1', 'LLM likely generates order_total instead of line_value'),

    ('L1-02', 'How many orders were placed this year?',
     "SELECT COUNT(*) as order_count FROM orders WHERE EXTRACT(YEAR FROM order_date) = EXTRACT(YEAR FROM CURRENT_DATE)",
     'E1', ''),

    ('L1-03', 'What is our average order value?',
     "SELECT AVG(line_value) as avg_order_value FROM orders",
     'E2', 'Could confuse SUM/AVG'),

    ('L1-04', 'How many unique customers made a purchase in the last 6 months?',
     "SELECT COUNT(DISTINCT customer_id) as unique_customers FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL 180 DAY",
     'E1', ''),

    ('L1-05', 'What is the total number of products in our catalog?',
     "SELECT COUNT(*) as product_count FROM products",
     '', 'Simple count - should work for all systems'),

    ('L1-06', 'What was our highest single-order revenue ever?',
     "SELECT MAX(line_value) as max_revenue FROM orders",
     'E1', 'line_value column name trap'),

    ('L1-07', 'How many orders were cancelled this year?',
     "SELECT COUNT(*) as cancelled_count FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = EXTRACT(YEAR FROM CURRENT_DATE)",
     'E1', ''),

    ('L1-08', 'What is our total revenue for Electronics products?',
     "SELECT SUM(o.line_value) as electronics_revenue FROM orders o JOIN products p ON o.product_id = p.id WHERE p.category = 'Electronics'",
     'E1', 'line_value + join needed'),

    ('L1-09', 'How many orders shipped in the last 30 days?',
     "SELECT COUNT(*) as shipped_count FROM orders WHERE ship_date >= CURRENT_DATE - INTERVAL 30 DAY AND ship_date IS NOT NULL",
     'E1', ''),

    ('L1-10', 'What is the revenue for the top-selling product?',
     "SELECT p.name, SUM(o.line_value) as rev FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.name ORDER BY rev DESC LIMIT 1",
     'E1', 'line_value + join + aggregation'),

    ('L1-11', 'How many new customers signed up this year?',
     "SELECT COUNT(*) as new_customers FROM customers WHERE EXTRACT(YEAR FROM created_at) = EXTRACT(YEAR FROM CURRENT_DATE)",
     'E1', ''),

    ('L1-12', 'What is our total cost of goods sold this year?',
     "SELECT SUM(p.cost_price * o.quantity) as cogs FROM orders o JOIN products p ON o.product_id = p.id WHERE EXTRACT(YEAR FROM o.order_date) = EXTRACT(YEAR FROM CURRENT_DATE)",
     'E1', 'cost_price column name + join'),

    ('L1-13', 'What percentage of orders were returned?',
     "SELECT ROUND(100.0 * COUNT(CASE WHEN status='returned' THEN 1 END) / COUNT(*), 2) as return_pct FROM orders",
     'E2', 'Percentage logic'),

    ('L1-14', 'How many distinct product categories do we sell?',
     "SELECT COUNT(DISTINCT category) as category_count FROM products",
     '', 'Simple query'),

    ('L1-15', 'What was our total revenue last year?',
     "SELECT SUM(line_value) as total_revenue FROM orders WHERE EXTRACT(YEAR FROM order_date) = EXTRACT(YEAR FROM CURRENT_DATE) - 1",
     'E1', 'line_value trap'),
]

for qid, prompt, sql, ec, note in l1_questions:
    questions.append(create_question(qid, 'L1', prompt, sql, ec, note))

print(f'✅ L1: {len(l1_questions)} questions created')

In [ ]:
# ═══════════════════════════════════════════════
# L2 — Multi-Table Join, Single Source (15 questions)
# ═══════════════════════════════════════════════

l2_questions = [
    ('L2-01', 'What is the revenue breakdown by product category?',
     "SELECT p.category, SUM(o.line_value) as revenue FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.category ORDER BY revenue DESC",
     'E1', 'line_value + join'),

    ('L2-02', 'Which customers placed more than 5 orders?',
     "SELECT c.name, COUNT(o.id) as order_count FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name HAVING COUNT(o.id) > 5 ORDER BY order_count DESC",
     'E3', 'Join + HAVING'),

    ('L2-03', 'Who are our top 10 customers by lifetime revenue?',
     "SELECT c.name, SUM(o.line_value) as lifetime_rev FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name ORDER BY lifetime_rev DESC LIMIT 10",
     'E1', 'line_value + join + limit'),

    ('L2-04', 'What is the average days between order and shipment per product category?',
     "SELECT p.category, AVG(o.ship_date - o.order_date) as avg_ship_days FROM orders o JOIN products p ON o.product_id = p.id WHERE o.ship_date IS NOT NULL GROUP BY p.category",
     'E2', 'Date arithmetic + AVG'),

    ('L2-05', 'Which product categories have more than 10 percent return rate?',
     "SELECT p.category, ROUND(100.0 * COUNT(CASE WHEN o.status='returned' THEN 1 END) / COUNT(*), 2) as return_rate FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.category HAVING ROUND(100.0 * COUNT(CASE WHEN o.status='returned' THEN 1 END) / COUNT(*), 2) > 10",
     'E2', 'Complex aggregation + HAVING'),

    ('L2-06', 'What is the revenue per customer segment?',
     "SELECT c.segment, SUM(o.line_value) as segment_revenue FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.segment ORDER BY segment_revenue DESC",
     'E3', 'segment column + join'),

    ('L2-07', 'Which sales rep closed the most orders?',
     "SELECT r.rep_name, COUNT(o.id) as order_count FROM orders o JOIN sales_reps r ON o.sales_rep_id = r.id GROUP BY r.rep_name ORDER BY order_count DESC LIMIT 1",
     'E3', 'sales_reps join'),

    ('L2-08', 'What is the monthly revenue trend for the last 12 months?',
     "SELECT DATE_TRUNC('month', order_date) as month, SUM(line_value) as monthly_revenue FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL 365 DAY GROUP BY month ORDER BY month",
     'E2', 'DATE_TRUNC + trend'),

    ('L2-09', 'Which products had zero orders in the last 60 days?',
     "SELECT p.name FROM products p WHERE p.id NOT IN (SELECT DISTINCT product_id FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL 60 DAY)",
     'E3', 'NOT IN subquery or LEFT JOIN'),

    ('L2-10', 'What is the average order value per customer segment?',
     "SELECT c.segment, AVG(o.line_value) as avg_order_value FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.segment",
     'E2', 'AVG + join'),

    ('L2-11', 'What is the revenue from completed orders versus pending orders?',
     "SELECT status, SUM(line_value) as revenue FROM orders WHERE status IN ('completed', 'pending') GROUP BY status",
     'E2', 'Status-based aggregation'),

    ('L2-12', 'Which product has the highest total quantity sold?',
     "SELECT p.name, SUM(o.quantity) as total_qty FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.name ORDER BY total_qty DESC LIMIT 1",
     'E3', 'quantity aggregation + join'),

    ('L2-13', 'What is the total discount given across all orders?',
     "SELECT SUM(line_value * discount_pct / 100.0) as total_discount FROM orders WHERE discount_pct > 0",
     'E2', 'Discount calculation'),

    ('L2-14', 'What is the average number of items per order by category?',
     "SELECT p.category, AVG(o.quantity) as avg_items FROM orders o JOIN products p ON o.product_id = p.id GROUP BY p.category",
     'E3', 'quantity + join'),

    ('L2-15', 'What percentage of revenue comes from our top 20 percent of customers?',
     "WITH customer_rev AS (SELECT customer_id, SUM(line_value) as rev FROM orders GROUP BY customer_id), ranked AS (SELECT rev, NTILE(5) OVER (ORDER BY rev DESC) as quintile FROM customer_rev) SELECT ROUND(100.0 * SUM(CASE WHEN quintile = 1 THEN rev END) / SUM(rev), 2) as top20_pct FROM ranked",
     'E2', 'Complex Pareto query'),
]

for qid, prompt, sql, ec, note in l2_questions:
    questions.append(create_question(qid, 'L2', prompt, sql, ec, note))

print(f'✅ L2: {len(l2_questions)} questions created')

In [ ]:
# ═══════════════════════════════════════════════
# L3 — Cross-Source Multi-Hop (10 questions)
# ═══════════════════════════════════════════════

l3_questions = [
    ('L3-01', 'Which customers with active CRM deals had delivery issues in the last 30 days?',
     "SELECT DISTINCT c.name, c.email FROM customers c JOIN sf_accounts sa ON c.email = sa.email_address JOIN sf_opportunities so ON sa.account_id = so.account_id JOIN orders o ON c.id = o.customer_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref WHERE so.stage NOT IN ('Closed Won', 'Closed Lost') AND ld.status_code IN (5, 6) AND CAST(REPLACE(ld.expected_date, '/', '-') AS DATE) >= CURRENT_DATE - INTERVAL 30 DAY",
     'E5', 'Three-source join: PG + SF + Logistics'),

    ('L3-02', 'What is the average deal size for customers whose deliveries were delayed?',
     "SELECT AVG(so.amount) as avg_deal_size FROM sf_opportunities so JOIN sf_accounts sa ON so.account_id = sa.account_id JOIN customers c ON c.email = sa.email_address JOIN orders o ON c.id = o.customer_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref WHERE ld.status_code = 5",
     'E5', 'SF + Logistics cross-source'),

    ('L3-03', 'Which of our top 10 revenue customers have an open CRM opportunity?',
     "WITH top10 AS (SELECT c.id, c.name, c.email, SUM(o.line_value) as rev FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.id, c.name, c.email ORDER BY rev DESC LIMIT 10) SELECT t.name, t.rev, so.deal_name, so.stage FROM top10 t JOIN sf_accounts sa ON t.email = sa.email_address JOIN sf_opportunities so ON sa.account_id = so.account_id WHERE so.stage NOT IN ('Closed Won', 'Closed Lost')",
     'E5', 'PG top customers + SF opportunities'),

    ('L3-04', 'What is the delivery success rate for Enterprise segment customers from CRM?',
     "SELECT ROUND(100.0 * COUNT(CASE WHEN ld.status_code = 4 THEN 1 END) / COUNT(*), 2) as success_rate FROM sf_accounts sa JOIN customers c ON c.email = sa.email_address JOIN orders o ON c.id = o.customer_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref WHERE sa.sf_segment = 'Enterprise'",
     'E5', 'SF segment + Logistics rates'),

    ('L3-05', 'Total revenue from customers whose CRM deals closed successfully?',
     "SELECT SUM(o.line_value) as total_revenue FROM orders o JOIN customers c ON o.customer_id = c.id JOIN sf_accounts sa ON c.email = sa.email_address JOIN sf_opportunities so ON sa.account_id = so.account_id WHERE so.stage = 'Closed Won'",
     'E5', 'PG revenue + SF deal status'),

    ('L3-06', 'Which customers have both high order value and a delayed shipment?',
     "SELECT DISTINCT c.name, SUM(o.line_value) as total_rev FROM customers c JOIN orders o ON c.id = o.customer_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref WHERE ld.status_code IN (5, 6) GROUP BY c.name HAVING SUM(o.line_value) > (SELECT AVG(line_value) * 2 FROM orders)",
     'E5', 'PG orders + Logistics delays'),

    ('L3-07', 'What is the on-time delivery rate by CRM deal stage?',
     "SELECT so.stage, ROUND(100.0 * COUNT(CASE WHEN ld.status_code = 4 THEN 1 END) / COUNT(*), 2) as ontime_rate FROM sf_opportunities so JOIN sf_accounts sa ON so.account_id = sa.account_id JOIN customers c ON c.email = sa.email_address JOIN orders o ON c.id = o.customer_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref GROUP BY so.stage",
     'E5', 'SF stages + Logistics rates'),

    ('L3-08', 'Which product categories have delivery complaints from high-value CRM accounts?',
     "SELECT p.category, COUNT(*) as complaint_count FROM products p JOIN orders o ON p.id = o.product_id JOIN logistics_deliveries ld ON o.shipping_id = ld.shipping_ref JOIN customers c ON o.customer_id = c.id JOIN sf_accounts sa ON c.email = sa.email_address WHERE ld.status_code IN (5, 6) AND sa.annual_revenue > 1000000 GROUP BY p.category ORDER BY complaint_count DESC",
     'E5', 'Three-source: PG products + Logistics + SF accounts'),

    ('L3-09', 'What is the average time from CRM deal close to first order?',
     "SELECT AVG(o.min_order_date - CAST(so.close_date AS DATE)) as avg_days FROM sf_opportunities so JOIN sf_accounts sa ON so.account_id = sa.account_id JOIN customers c ON c.email = sa.email_address JOIN (SELECT customer_id, MIN(order_date) as min_order_date FROM orders GROUP BY customer_id) o ON c.id = o.customer_id WHERE so.stage = 'Closed Won'",
     'E5', 'SF deal close + PG first order'),

    ('L3-10', 'Which logistics carrier has the worst on-time rate for our top revenue customers?',
     "WITH top_customers AS (SELECT c.id FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.id ORDER BY SUM(o.line_value) DESC LIMIT 100) SELECT ld.carrier_name, ROUND(100.0 * COUNT(CASE WHEN ld.status_code = 4 THEN 1 END) / COUNT(*), 2) as ontime_rate FROM logistics_deliveries ld JOIN orders o ON ld.shipping_ref = o.shipping_id WHERE o.customer_id IN (SELECT id FROM top_customers) GROUP BY ld.carrier_name ORDER BY ontime_rate ASC LIMIT 1",
     'E5', 'PG revenue ranking + Logistics carrier rates'),
]

for qid, prompt, sql, ec, note in l3_questions:
    questions.append(create_question(qid, 'L3', prompt, sql, ec, note))

print(f'✅ L3: {len(l3_questions)} questions created')

In [ ]:
# ═══════════════════════════════════════════════
# L4 — Unstructured + Structured RAG (10 questions)
# ═══════════════════════════════════════════════
# Note: L4 uses synthetic email/ticket data. Gold results are based on
# structured queries over the synthetic unstructured data stored as JSON.

l4_questions = [
    ('L4-01', 'Summarize delivery complaint themes for our top 10 revenue customers.',
     "SELECT c.name, SUM(o.line_value) as rev FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name ORDER BY rev DESC LIMIT 10",
     'E4', 'Requires RAG over emails + PG revenue'),

    ('L4-02', 'Which product categories have the most negative sentiment in support tickets?',
     "SELECT p.category, COUNT(*) as negative_count FROM products p JOIN orders o ON p.id = o.product_id GROUP BY p.category ORDER BY negative_count DESC",
     'E4', 'Requires sentiment analysis from tickets'),

    ('L4-03', 'What common complaints appear for accounts with delayed shipments?',
     "SELECT ld.carrier_name, COUNT(*) as delay_count FROM logistics_deliveries ld WHERE ld.status_code = 5 GROUP BY ld.carrier_name ORDER BY delay_count DESC",
     'E4', 'Requires text analysis from SF notes + Logistics'),

    ('L4-04', 'Find customers who mentioned competitor products in support interactions.',
     "SELECT DISTINCT customer_id FROM orders WHERE customer_id IS NOT NULL LIMIT 20",
     'E4', 'Requires NER/keyword extraction from support chat'),

    ('L4-05', 'What were the top 3 reasons for returns based on return notes?',
     "SELECT reason, COUNT(*) as count FROM returns GROUP BY reason ORDER BY count DESC LIMIT 3",
     'E4', 'Structured return reasons available'),

    ('L4-06', 'Which sales reps have customers with the most positive feedback?',
     "SELECT r.rep_name, COUNT(o.id) as completed_orders FROM sales_reps r JOIN orders o ON r.id = o.sales_rep_id WHERE o.status = 'completed' GROUP BY r.rep_name ORDER BY completed_orders DESC LIMIT 5",
     'E4', 'Requires sentiment from deal notes'),

    ('L4-07', 'Summarize key themes in feedback for Enterprise segment accounts.',
     "SELECT c.segment, COUNT(*) as order_count FROM customers c JOIN orders o ON c.id = o.customer_id WHERE c.segment = 'Enterprise' GROUP BY c.segment",
     'E4', 'Requires text analysis from onboarding surveys'),

    ('L4-08', 'Which logistics partners have the most damaged goods complaints?',
     "SELECT ld.carrier_name, COUNT(*) as issue_count FROM logistics_deliveries ld WHERE ld.status_code IN (5, 6) GROUP BY ld.carrier_name ORDER BY issue_count DESC",
     'E4', 'Requires text analysis from damage reports'),

    ('L4-09', 'What do churned customers say were their top reasons for leaving?',
     "SELECT reason, COUNT(*) as count FROM returns GROUP BY reason ORDER BY count DESC",
     'E4', 'Requires exit survey analysis'),

    ('L4-10', 'Find orders where customer complaints mention a product defect.',
     "SELECT o.id, p.name FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'returned' LIMIT 20",
     'E4', 'Requires NER from complaint text + order matching'),
]

for qid, prompt, sql, ec, note in l4_questions:
    questions.append(create_question(qid, 'L4', prompt, sql, ec, note))

print(f'✅ L4: {len(l4_questions)} questions created')

In [ ]:
# Save all questions
for q in questions:
    filepath = f"{PROJECT_DIR}/eval/questions/{q['id']}.json"
    with open(filepath, 'w') as f:
        json.dump(q, f, indent=2, default=str)

print(f'\n✅ Total questions saved: {len(questions)}')
print(f'  L1: {sum(1 for q in questions if q["tier"]=="L1")}')
print(f'  L2: {sum(1 for q in questions if q["tier"]=="L2")}')
print(f'  L3: {sum(1 for q in questions if q["tier"]=="L3")}')
print(f'  L4: {sum(1 for q in questions if q["tier"]=="L4")}')

# Verify gold SQL success rate
success = sum(1 for q in questions if q['gold_result'] is not None)
print(f'\n📊 Gold SQL success: {success}/{len(questions)} ({100*success/len(questions):.0f}%)')

## ✅ Data Environment Complete

**Next step:** Run `04_experiment_runner.ipynb` to execute all 8 experiments.